# TXLA virtual Icechunk: smoke test

Writes a **temporary local** repository (never `txla_icechunks/`) from 4 source files chosen to exercise the edge cases found in the audit:

| file | why |
|---|---|
| `2026-08-28-213243` | short run: 42 of 96 records |
| `2026-08-29-214025` | normal 96-record run |
| `2026-08-30-114753` | 1-record file, same forecast start as the next file → must be excluded |
| `2026-08-31-171807` | full run for 2026-08-30, supersedes the 1-record file |

It then reopens the store through the consumer read path, reads real values, and compares them with the source file read directly.

In [1]:
import sys, time, tempfile, platform
import numpy as np, xarray as xr, zarr, icechunk, virtualizarr, obstore, netCDF4
from txla_roms_icechunk import build as tb
for m in (icechunk, virtualizarr, xr, zarr, obstore, netCDF4, np):
    print(f"{m.__name__:14s}{m.__version__}")
print("python        ", platform.python_version())

icechunk      2.3.1
virtualizarr  2.8.0
xarray        2026.9.0
zarr          3.4.2
obstore       0.12.1
netCDF4       1.7.5
numpy         2.5.5
python         3.13.18


In [2]:
names = ["2026/08/txla2_his_f_latest_z.2026-08-28-213243.nc",
         "2026/08/txla2_his_f_latest_z.2026-08-29-214025.nc",
         "2026/08/txla2_his_f_latest_z.2026-08-30-114753.nc",
         "2026/08/txla2_his_f_latest_z.2026-08-31-171807.nc"]
urls = [tb.SOURCE_PREFIX + n for n in names]
urls

['https://hriai01.tamucc.edu/data/MASTR_MODELS/txla/history/2026/08/txla2_his_f_latest_z.2026-08-28-213243.nc',
 'https://hriai01.tamucc.edu/data/MASTR_MODELS/txla/history/2026/08/txla2_his_f_latest_z.2026-08-29-214025.nc',
 'https://hriai01.tamucc.edu/data/MASTR_MODELS/txla/history/2026/08/txla2_his_f_latest_z.2026-08-30-114753.nc',
 'https://hriai01.tamucc.edu/data/MASTR_MODELS/txla/history/2026/08/txla2_his_f_latest_z.2026-08-31-171807.nc']

## Byte-range access (anonymous HTTPS, no credentials needed)

In [3]:
import urllib.request
for u in urls:
    req = urllib.request.Request(u, headers={"Range": "bytes=0-3"})
    with urllib.request.urlopen(req, timeout=60) as r:
        assert r.status == 206, r.status
        assert r.read() == b"CDF\x01", "not NetCDF-3 classic"
print("206 + CDF\\x01 for all", len(urls))

206 + CDF\x01 for all 4


## Open each file virtually (header + time only), select runs, check compatibility

In [4]:
reg = tb.make_registry()
t = time.time(); runs = tb.open_runs(urls, reg); print(f"opened {len(runs)} in {time.time()-t:.1f}s")
for r in runs: print(r.archived, r.frt, r.n, "records")
kept, excluded = tb.select_runs(runs)
print("kept:", [r.archived for r in kept]); print("excluded:", excluded)
assert [r.archived for r in kept] == ["2026-08-28-213243", "2026-08-29-214025", "2026-08-31-171807"]
data_vars = tb.check_compatible(kept); print(data_vars)
grid = tb.load_grid(kept, reg)

opened 4 in 2.3s
2026-08-28-213243 2026-08-28T00:00:00.000000000 42 records
2026-08-29-214025 2026-08-29T00:00:00.000000000 96 records
2026-08-30-114753 2026-08-30T00:00:00.000000000 1 records
2026-08-31-171807 2026-08-30T00:00:00.000000000 96 records
kept: ['2026-08-28-213243', '2026-08-29-214025', '2026-08-31-171807']
excluded: [('https://hriai01.tamucc.edu/data/MASTR_MODELS/txla/history/2026/08/txla2_his_f_latest_z.2026-08-30-114753.nc', 'superseded by later-archived run 2026-08-31-171807 for the same forecast_reference_time (1 records)')]
['zeta', 'air_u', 'air_v', 'sustr', 'svstr', 'u', 'v', 'temp', 'salt', 'O2', 'pH', 'omega']


## Assemble the two groups

In [5]:
fc = tb.build_forecast(kept, excluded, grid)
best = tb.build_best(kept, excluded, grid)
print(fc); print(best)
print("best missing hours:", best.attrs["missing_hours"])
for v in ["temp", "zeta"]:
    print(v, "attrs:", fc[v].attrs)

<xarray.Dataset> Size: 29GB
Dimensions:                  (forecast_reference_time: 3, lead: 96, Y: 190,
                              X: 670, Depth: 28)
Coordinates:
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 24B 202...
  * lead                     (lead) timedelta64[ns] 768B 01:00:00 ... 4 days ...
    valid_time               (forecast_reference_time, lead) datetime64[ns] 2kB ...
    lon                      (Y, X) float64 1MB -95.49 -95.49 ... -88.88 -88.87
    lat                      (Y, X) float64 1MB 23.13 23.16 ... 30.84 30.85
  * Depth                    (Depth) float64 224B 1.0 3.0 5.0 ... 900.0 1e+03
Dimensions without coordinates: Y, X
Data variables:
    zeta                     (forecast_reference_time, lead, Y, X) float32 147MB ManifestArray<shape=(3, 96, 190, 670), dtype=float32, chunks=(1, 1, 190, 6...
    air_u                    (forecast_reference_time, lead, Y, X) float32 147MB ManifestArray<shape=(3, 96, 190, 670), dtype=float32, chunks=

## Write to a temporary local repository and commit

In [6]:
tmp = tempfile.mkdtemp(prefix="txla_smoke_")
repo = tb.open_or_create(tmp)
snap = tb.write(repo, fc, best, "smoke test: 3 runs")
print(tmp, snap)

/tmp/txla_smoke_l9dbacu7 Y98KVDD2S6PTZAEAZR9G


## Reopen through the consumer read path (fresh `Repository.open`, read-only session)

In [7]:
zarr.config.set({"async.concurrency": 32})  # modest: the source is one university server
F = tb.open_readonly(tmp, "forecast")
B = tb.open_readonly(tmp, "best")
print(F); print(B)
assert F.sizes == {"forecast_reference_time": 3, "lead": 96, "Depth": 28, "Y": 190, "X": 670}
assert list(F.forecast_reference_time.values.astype("datetime64[D]").astype(str)) == ["2026-08-28", "2026-08-29", "2026-08-30"]
assert int(F.lead[0] / np.timedelta64(1, "h")) == 1 and int(F.lead[-1] / np.timedelta64(1, "h")) == 96
assert B.time.diff("time").min() == B.time.diff("time").max() == np.timedelta64(1, "h")
assert str(B.time.values[0])[:13] == "2026-08-28T01" and str(B.time.values[-1])[:13] == "2026-09-03T00"
for v in data_vars:
    assert F[v].dtype == np.float32
    assert F[v].encoding["chunks"][:2] == (1, 1), F[v].encoding["chunks"]
print("structure OK")

<xarray.Dataset> Size: 29GB
Dimensions:                  (forecast_reference_time: 3, lead: 96, Depth: 28,
                              Y: 190, X: 670)
Coordinates:
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 24B 202...
  * lead                     (lead) timedelta64[ns] 768B 01:00:00 ... 4 days ...
    valid_time               (forecast_reference_time, lead) datetime64[ns] 2kB ...
  * Depth                    (Depth) float64 224B 1.0 3.0 5.0 ... 900.0 1e+03
    lat                      (Y, X) float64 1MB ...
    lon                      (Y, X) float64 1MB ...
Dimensions without coordinates: Y, X
Data variables:
    O2                       (forecast_reference_time, lead, Depth, Y, X) float32 4GB ...
    air_v                    (forecast_reference_time, lead, Y, X) float32 147MB ...
    pH                       (forecast_reference_time, lead, Depth, Y, X) float32 4GB ...
    svstr                    (forecast_reference_time, lead, Y, X) float32 147MB ...
   

## Read real values and compare with the source file read directly

In [8]:
def src(name, var, rec):
    with netCDF4.Dataset(tb.SOURCE_PREFIX + name + "#mode=bytes") as d:
        return d[var][rec].filled(np.nan)

# forecast: run 2026-08-29, lead 24 h -> record 23 of 2026-08-29-214025
t = time.time(); a = F.temp.sel(forecast_reference_time="2026-08-29", lead=np.timedelta64(24, "h")).values; dt = time.time()-t
b = src(names[1], "temp", 23)
np.testing.assert_array_equal(a, b); print(f"temp field (14 MB) equal, {dt:.1f}s, NaN fraction {np.isnan(a).mean():.2f}")

# best: 2026-08-31T05 must come from the 2026-08-30 run (file 08-31-171807), lead 29 -> record 28
p = B.sel(time="2026-08-31T05:00")
assert str(p.forecast_reference_time.values)[:10] == "2026-08-30" and int(p.lead / np.timedelta64(1, "h")) == 29
np.testing.assert_array_equal(p.zeta.values, src(names[3], "zeta", 28)); print("best zeta matches newest run")

# missing chunks: the 42-record run has no lead 43..96 -> all NaN
assert np.isnan(F.zeta.sel(forecast_reference_time="2026-08-28", lead=np.timedelta64(50, "h")).values).all()
print("short run reads as NaN beyond its last record")

temp field (14 MB) equal, 1.0s, NaN fraction 0.63


best zeta matches newest run
short run reads as NaN beyond its last record


## A query shaped like the intended science: a point time series from `/best`

In [9]:
t = time.time(); ts = B.zeta.isel(Y=95, X=335).values; dt = time.time()-t
print(f"zeta point series, {ts.size} hours: {dt:.1f}s ({ts.size} x 0.5 MB requests)")
t = time.time(); tp = B.temp.isel(Y=95, X=335, Depth=0, time=slice(0, 24)).values; dt = time.time()-t
print(f"surface temp point series, 24 hours: {dt:.1f}s (24 x 14.3 MB requests)")
print(np.round(tp, 2))

zeta point series, 144 hours: 2.4s (144 x 0.5 MB requests)


surface temp point series, 24 hours: 2.5s (24 x 14.3 MB requests)
[31.95 31.87 31.74 31.61 31.51 31.43 31.37 31.33 31.31 31.29 31.29 31.29
 31.32 31.32 31.31 31.31 31.33 31.35 31.39 31.45 31.55 31.63 31.63 31.61]


## Still unknown before the full build

- **Point time series are expensive for 3-D variables.** Every chunk is a whole 670×190×28 field (14.3 MB); one point for one hour fetches all of it. Nothing metadata-only can fix that.
- **The source server's tolerance for load** is not known; readers should keep concurrency modest.
- **Stability of the URLs**: the store breaks if TAMU moves, renames or rewrites a history file. Not confirmed with TAMU.
- **No CORS on the source host**, so browser readers will not get data.

In [10]:
import shutil; shutil.rmtree(tmp); print("removed", tmp)

removed /tmp/txla_smoke_l9dbacu7
